# NB06: Fitness Browser Cross-Reference

**Goal**: Test H6 — do AMR genes impose measurable fitness costs? For the ~48 Fitness Browser organisms, identify which carry AMR genes in the pangenome, then extract fitness effects.

**Caveats**:
- Only 48 FB organisms, not all will have AMR genes in bakta_amr
- FB fitness values are STRING type — must CAST before numeric operations
- This is exploratory — small sample size limits statistical power

**Depends on**: NB01 outputs + Spark session

**Outputs**:
- `../data/amr_fitness.csv`
- `../figures/amr_fitness_*.png`

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

try:
    spark = get_spark_session()
except NameError:
    sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', '..', 'scripts'))
    from get_spark_session import get_spark_session
    spark = get_spark_session()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')

plt.rcParams.update({
    'figure.figsize': (12, 6), 'figure.dpi': 150, 'font.size': 11,
    'axes.titlesize': 13, 'savefig.bbox': 'tight', 'savefig.dpi': 150,
})

amr = pd.read_csv(DATA_DIR / 'amr_census.csv')
print(f"Loaded {len(amr):,} AMR clusters")

## 1. Find FB Organisms with AMR Genes

Use the existing FB-pangenome link table from `conservation_vs_fitness` to map FB organisms to pangenome species, then check which have AMR annotations.

In [ ]:
# Get FB organisms and their pangenome links
# The conservation_vs_fitness project established the link table
fb_orgs = spark.sql("""
    SELECT DISTINCT orgId, organism
    FROM kescience_fitnessbrowser.organism
""").toPandas()
print(f"Fitness Browser organisms: {len(fb_orgs):,}")

# Get the FB-pangenome link (locusId → gene_cluster_id mapping)
# This requires joining through gene sequences — use the existing link if available
# For now, use a simpler approach: match FB organism names to GTDB species
fb_species_match = spark.sql("""
    SELECT DISTINCT o.orgId, o.organism, g.gtdb_species_clade_id
    FROM kescience_fitnessbrowser.organism o
    JOIN kescience_fitnessbrowser.gene fg ON o.orgId = fg.orgId
    JOIN kbase_ke_pangenome.genome gn ON fg.scaffoldId = gn.genome_id
    GROUP BY o.orgId, o.organism, g.gtdb_species_clade_id
""")

# If that fails (scaffoldId may not match genome_id), fall back to name matching
# Let's try a simpler approach — check which AMR species appear in FB
amr_species_set = set(amr['gtdb_species_clade_id'].unique())

# Get FB species names and try to match
print("\nAttempting to match FB organisms to pangenome species...")
print("FB organisms:")
for _, row in fb_orgs.iterrows():
    print(f"  {row['orgId']}: {row['organism']}")

In [ ]:
# Use the existing FB-pangenome link table from conservation_vs_fitness
# This was built by matching FB gene sequences to pangenome cluster representatives
import os

link_path = os.path.join('..', '..', 'conservation_vs_fitness', 'data', 'fb_pangenome_links.csv')
if os.path.exists(link_path):
    fb_links = pd.read_csv(link_path)
    print(f"Loaded FB-pangenome link table: {len(fb_links):,} links")
else:
    # Try the lakehouse version or build from scratch
    print(f"Link table not found at {link_path}")
    print("Will attempt to build links using gene name matching.")
    print("For full analysis, run conservation_vs_fitness NB01 first or download from lakehouse.")
    fb_links = None

# If we have links, find AMR genes in FB organisms
if fb_links is not None:
    # Merge: FB links × AMR census
    fb_amr = fb_links.merge(amr[['gene_cluster_id', 'amr_gene', 'amr_product', 'mechanism',
                                  'conservation_class', 'identity']],
                            on='gene_cluster_id', how='inner')
    print(f"\nFB genes with AMR annotations: {len(fb_amr):,}")
    print(f"Distinct FB organisms with AMR genes: {fb_amr['orgId'].nunique()}")
    print(f"Distinct AMR genes in FB: {fb_amr['amr_gene'].nunique()}")
else:
    # Fallback: count AMR species that overlap with FB organism names
    print("\nFalling back to species-name matching (less precise)...")
    print("This will be refined when the link table is available.")

In [ ]:
# If we have FB-AMR matches, extract fitness data
if fb_links is not None and len(fb_amr) > 0:
    # Get fitness data for AMR genes
    # NOTE: FB fitness values are STRING type — must CAST
    fb_amr_orgs = fb_amr['orgId'].unique().tolist()
    fb_amr_loci = fb_amr['locusId'].unique().tolist()
    
    # Create temp views
    spark.createDataFrame([(x,) for x in fb_amr_loci], ['locusId']).createOrReplaceTempView('amr_loci')
    
    amr_fitness = spark.sql("""
        SELECT gf.orgId, gf.locusId, 
               CAST(gf.fit as DOUBLE) as fitness,
               CAST(gf.t as DOUBLE) as t_score
        FROM kescience_fitnessbrowser.genefitness gf
        JOIN amr_loci al ON gf.locusId = al.locusId
    """).toPandas()
    
    print(f"Fitness measurements for AMR genes: {len(amr_fitness):,}")
    print(f"Organisms: {amr_fitness['orgId'].nunique()}")
    print(f"Genes: {amr_fitness['locusId'].nunique()}")
    
    # Summary: are AMR genes costly?
    print(f"\n=== AMR Gene Fitness Summary ===")
    print(f"Mean fitness: {amr_fitness['fitness'].mean():.4f}")
    print(f"Median fitness: {amr_fitness['fitness'].median():.4f}")
    print(f"% with sick phenotype (fitness < -1): {(amr_fitness['fitness'] < -1).mean()*100:.1f}%")
    print(f"% with strong sick (fitness < -2): {(amr_fitness['fitness'] < -2).mean()*100:.1f}%")
    
    # Compare to non-AMR genes: get baseline for same organisms
    print("\nExtracting baseline fitness for comparison (same organisms, non-AMR genes)...")
    baseline_fitness = spark.sql(f"""
        SELECT CAST(fit as DOUBLE) as fitness
        FROM kescience_fitnessbrowser.genefitness
        WHERE orgId IN ('{"','".join(fb_amr_orgs)}')
        AND locusId NOT IN (SELECT locusId FROM amr_loci)
    """).toPandas()
    
    print(f"Baseline measurements: {len(baseline_fitness):,}")
    print(f"Baseline mean fitness: {baseline_fitness['fitness'].mean():.4f}")
    
    # Mann-Whitney test
    u_stat, p_mw = stats.mannwhitneyu(amr_fitness['fitness'], baseline_fitness['fitness'])
    print(f"\nMann-Whitney (AMR vs non-AMR fitness): U={u_stat:.0f}, p={p_mw:.2e}")
    
    # Save
    amr_fitness.to_csv(DATA_DIR / 'amr_fitness.csv', index=False)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(amr_fitness['fitness'], bins=100, alpha=0.7, color='#e74c3c', label='AMR genes', density=True)
    axes[0].hist(baseline_fitness['fitness'], bins=100, alpha=0.5, color='#95a5a6', label='Non-AMR', density=True)
    axes[0].set_xlabel('Fitness Score')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Fitness Distribution: AMR vs Non-AMR Genes')
    axes[0].legend()
    axes[0].set_xlim(-6, 4)
    
    # Per-organism AMR fitness summary
    org_summary = amr_fitness.groupby('orgId')['fitness'].agg(['mean', 'median', 'count']).round(3)
    org_summary = org_summary.sort_values('mean')
    axes[1].barh(range(len(org_summary)), org_summary['mean'], color='#e74c3c')
    axes[1].set_yticks(range(len(org_summary)))
    axes[1].set_yticklabels(org_summary.index, fontsize=8)
    axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Mean Fitness of AMR Genes')
    axes[1].set_title('AMR Gene Fitness by Organism')
    
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'amr_fitness_distribution.png')
    plt.show()
else:
    print("Skipping fitness analysis — no FB-pangenome link table available.")
    print("To enable this analysis, run the conservation_vs_fitness project first.")